# 📚 Building a Prompt Library

**Day 3 — Notebook 4 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- How to build a reusable prompt library for common business tasks
- Categorization, summarization, extraction, and QA templates
- A `PromptLibrary` class you can import in future notebooks

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Building the Prompt Library

A prompt library is a collection of **tested, reusable prompt templates** for common tasks. Let's build one.

In [ ]:
class PromptLibrary:
    """A collection of reusable prompt templates."""
    
    def __init__(self, client, model_id: str):
        self.client = client
        self.model_id = model_id
    
    def _call(self, prompt: str, system: str = None, temp: float = 0.7) -> str:
        """Internal helper to call the model."""
        config = types.GenerateContentConfig(temperature=temp)
        if system:
            config.system_instruction = system
        response = self.client.models.generate_content(
            model=self.model_id, contents=prompt, config=config,
        )
        return response.text

    # --- Summarization Templates ---
    
    def summarize(self, text: str, length: str = "3 sentences", 
                  audience: str = "general") -> str:
        """Summarize text for a specific audience."""
        return self._call(f"""Summarize the following in {length} for a {audience} audience.

<text>
{text}
</text>""")
    
    def bullet_points(self, text: str, max_points: int = 5) -> str:
        """Extract key points as bullet points."""
        return self._call(f"""Extract the {max_points} most important points from this text.
Return as a bullet-point list.

<text>
{text}
</text>""")

    # --- Classification Templates ---
    
    def classify(self, text: str, categories: list[str]) -> str:
        """Classify text into one of the given categories."""
        cats = ", ".join(categories)
        return self._call(
            f"Classify this text into exactly one category: {cats}\n\nText: {text}\n\nCategory:",
            temp=0
        ).strip()
    
    def sentiment(self, text: str) -> str:
        """Analyze sentiment of text."""
        return self._call(
            f"Classify the sentiment as Positive, Negative, or Neutral. Reply with one word only.\n\nText: {text}\n\nSentiment:",
            temp=0
        ).strip()

    # --- Extraction Templates ---
    
    def extract_entities(self, text: str) -> str:
        """Extract named entities from text."""
        return self._call(f"""Extract all named entities from this text.
Categorize them as: Person, Organization, Location, Date, or Other.

<text>
{text}
</text>""")
    
    def extract_action_items(self, text: str) -> str:
        """Extract action items from meeting notes or emails."""
        return self._call(f"""Extract all action items from this text.
For each, list: WHO needs to do WHAT by WHEN (if mentioned).

<text>
{text}
</text>""")

    # --- Q&A Templates ---
    
    def answer_from_context(self, question: str, context: str) -> str:
        """Answer a question based only on the provided context."""
        return self._call(f"""Answer the question based ONLY on the context below.
If the answer isn't in the context, say "I cannot find this information in the provided context."

<context>
{context}
</context>

Question: {question}
Answer:""", temp=0)

    # --- Writing Templates ---
    
    def rewrite(self, text: str, style: str = "professional") -> str:
        """Rewrite text in a different style."""
        return self._call(f"""Rewrite the following text in a {style} style.
Keep the meaning the same.

<text>
{text}
</text>""")


# Initialize the library
prompts = PromptLibrary(client, MODEL_ID)
print("✅ Prompt Library initialized!")

---

## 2. Using the Library

In [ ]:
# Summarization
article = """
Google DeepMind's latest research shows that large language models can be 
significantly improved through careful prompt engineering without any additional 
training. Their study found that structured prompts with clear instructions, 
examples, and constraints led to a 40% improvement in task accuracy across 
benchmarks. The team recommends using system instructions, few-shot examples, 
and output schemas as standard practice for production applications.
"""

print("📄 Summary:")
print(prompts.summarize(article, length="2 sentences", audience="executive"))

In [ ]:
# Classification
texts = [
    "My server keeps crashing every 30 minutes",
    "I'd like to upgrade my subscription plan",
    "Your product changed my workflow completely!",
]

categories = ["Technical Issue", "Billing", "Feedback", "General Inquiry"]

for text in texts:
    category = prompts.classify(text, categories)
    sentiment = prompts.sentiment(text)
    print(f"Text: {text}")
    print(f"  Category: {category} | Sentiment: {sentiment}")

In [ ]:
# Context-grounded Q&A
company_policy = """
Work from home policy: Employees may work remotely up to 3 days per week.
All remote work must be approved by the direct manager. Employees must be 
available during core hours (10 AM - 3 PM EST). VPN must be used for all 
remote access to company systems.
"""

questions = [
    "How many days can I work from home?",
    "Do I need VPN for remote work?",
    "What is the company's travel reimbursement policy?",  # Not in context!
]

for q in questions:
    answer = prompts.answer_from_context(q, company_policy)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

---

## 🏋️ Exercise 1: Extend the Library

Add two new methods to the PromptLibrary class.

In [ ]:
# Exercise 1: Add new methods

# TODO 1: Add a `translate` method that translates text to a target language
# TODO 2: Add a `generate_test_data` method that generates sample data 
#          given a schema description (e.g., "5 customer records with name, email, purchase_amount")

# Test your new methods:
# prompts.translate("Hello, how are you?", target_language="Spanish")
# prompts.generate_test_data("3 employee records with name, department, and salary")

---

## 🏋️ Exercise 2: Processing Pipeline

Chain multiple library functions together to process a document.

In [ ]:
# Exercise 2: Document processing pipeline
# TODO: Given meeting notes, extract:
# 1. Summary (summarize)
# 2. Sentiment (sentiment)
# 3. Action items (extract_action_items)
# 4. Named entities (extract_entities)

meeting_notes = """
Sprint retrospective - March 15, 2024
Attendees: Alice Chen, Bob Singh, Carol Martinez

Alice presented the Q1 results - we exceeded our target by 15%. The new 
recommendation engine launched successfully with 98% uptime. Bob raised 
concerns about the increasing technical debt in the payment service. 
Carol agreed and suggested we allocate 20% of next sprint to refactoring.

Action items:
- Bob to create a tech debt backlog by Friday
- Alice to present Q1 results to leadership on Monday
- Carol to update sprint planning template by Wednesday
"""

# TODO: Build a pipeline that runs all 4 analyses
# Print results in a structured format

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Google Prompting Strategies | Docs | [ai.google.dev/docs/prompting-strategies](https://ai.google.dev/gemini-api/docs/prompting-strategies) |
| Prompt Engineering for Enterprise | Video | [Google Cloud YouTube](https://www.youtube.com/watch?v=jC4v5AS4RIM) |

---

🎉 **Congratulations!** You've completed Day 3. You now have advanced techniques and a reusable prompt library.

**Tomorrow:** [Day 4 — Hallucinations, Grounding & Generation Control](../Day_04_Hallucinations_and_Grounding/README.md)